In [ ]:
# ============================================================
# BRANCH B — STRUCTURAL CONVERSION
# D2 — VINCI Consolidated Income Statement 2024
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch B: Structural Conversion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!pip -q install pymupdf

import json
import hashlib
import re
import fitz
from pathlib import Path
from google.colab import files

DOCUMENT_ID = "D2"
DOCUMENT_NAME = "VINCI Consolidated Income Statement 2024"

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

CONVERSION_METHOD = (
    "PyMuPDF ordered text-block structural reconstruction"
)

EXPECTED_SOURCE_FORMAT = ".pdf"
EXPECTED_SOURCE_SHA256 = "6ea8c09b7e06f328ded876b76e55a11a2b84d56d7dff3e8a4bfd4ecfb3605667"
EXPECTED_PAGE_COUNT = 1
EXPECTED_RECORD_COUNT = 22

EXPECTED_FIELDS = [
    "Line Item",
    "Unit",
    "Value 2024",
    "Value 2023"
]

NUMERIC_FIELDS = [
    "Value 2024",
    "Value 2023"
]

ALLOWED_UNITS = [
    "EUR millions",
    "EUR"
]

OUTPUT_DIR = Path("outputs_D2_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected records:", EXPECTED_RECORD_COUNT)


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Upload exactly one source PDF for D2.")

SOURCE_FILE = Path(next(iter(uploaded)))

print("Loaded source:", SOURCE_FILE)


In [ ]:
# ============================================================
# 2. Verify source format and SHA-256 identity
# ============================================================

if SOURCE_FILE.suffix.lower() != EXPECTED_SOURCE_FORMAT:
    raise ValueError(
        f"Expected a {EXPECTED_SOURCE_FORMAT} source, "
        f"received {SOURCE_FILE.suffix}"
    )

def calculate_sha256(path, chunk_size=8192):
    sha256 = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            sha256.update(chunk)
    return sha256.hexdigest()

SOURCE_SHA256 = calculate_sha256(SOURCE_FILE)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

print("Observed source SHA-256:", SOURCE_SHA256)
print("Matches frozen D2 source:", SOURCE_HASH_MATCH)

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded PDF does not match the frozen D2 source identity."
    )


In [ ]:
# ============================================================
# 3. Inspect source-document properties
# ============================================================

pdf_document = fitz.open(SOURCE_FILE)

PAGE_COUNT = len(pdf_document)

page_character_counts = [
    len(page.get_text("text").strip())
    for page in pdf_document
]

PDF_MACHINE_READABLE = all(
    count > 0
    for count in page_character_counts
)

source_properties = {
    "page_count": PAGE_COUNT,
    "expected_page_count": EXPECTED_PAGE_COUNT,
    "page_count_matches": PAGE_COUNT == EXPECTED_PAGE_COUNT,
    "machine_readable_text_detected": PDF_MACHINE_READABLE,
    "characters_detected_per_page": page_character_counts
}

print(json.dumps(source_properties, indent=2))

if PAGE_COUNT != EXPECTED_PAGE_COUNT:
    raise ValueError("Unexpected D2 page count.")

if not PDF_MACHINE_READABLE:
    raise ValueError(
        "D2 is expected to be machine-readable. "
        "OCR is not permitted in Branch B for this document."
    )


## D2 structural-conversion rule

The source PDF encodes the income statement as a sequence of horizontal text blocks. Each data-row block contains one visible line-item label followed by its 2024 and 2023 values. Branch B makes this existing row/column relationship explicit as a Markdown table.

This is a deterministic structural reconstruction from the PDF text blocks. It does **not** use the Stage 1 reference values to populate or repair the representation.

In [ ]:
# ============================================================
# 4. Recover source text blocks in physical reading order
# ============================================================

page = pdf_document[0]

raw_blocks = page.get_text("blocks")

ordered_blocks = sorted(
    raw_blocks,
    key=lambda block: (
        round(block[1], 3),
        round(block[0], 3)
    )
)

block_audit = []

for i, block in enumerate(ordered_blocks):
    text = block[4].strip()

    if text:
        block_audit.append({
            "block_index": i,
            "x0": float(block[0]),
            "y0": float(block[1]),
            "x1": float(block[2]),
            "y1": float(block[3]),
            "text": text
        })

print("Non-empty source text blocks:", len(block_audit))

for item in block_audit:
    print(
        item["block_index"],
        round(item["y0"], 1),
        repr(item["text"][:140])
    )


In [ ]:
# ============================================================
# 5. Deterministically identify title/header, data rows and footnote
# ============================================================

def nonempty_lines(text):
    return [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

header_block = None
footnote_block = None
row_blocks = []

for item in block_audit:
    lines = nonempty_lines(item["text"])

    if (
        "(in € millions)" in item["text"]
        and "2024" in item["text"]
        and "2023" in item["text"]
    ):
        header_block = item
        continue

    if "(*) Excluding concession subsidiaries" in item["text"]:
        footnote_block = item
        continue

    # A D2 data-row block contains a label plus the 2024 and 2023 values.
    if len(lines) == 3:
        row_blocks.append(item)

if header_block is None:
    raise ValueError("Could not identify the D2 table header block.")

if footnote_block is None:
    raise ValueError("Could not identify the D2 footnote block.")

print("Recovered row-like blocks:", len(row_blocks))

if len(row_blocks) != EXPECTED_RECORD_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_RECORD_COUNT} D2 table rows, "
        f"recovered {len(row_blocks)}."
    )


In [ ]:
# ============================================================
# 6. Reconstruct the explicit table structure
# ============================================================

converted_rows = []

for item in row_blocks:
    lines = nonempty_lines(item["text"])

    converted_rows.append({
        "Line Item": lines[0],
        "2024": lines[1],
        "2023": lines[2],
        "_source_block_index": item["block_index"],
        "_source_y0": item["y0"]
    })

print("Converted rows:", len(converted_rows))

for row in converted_rows[:5]:
    print(row)


In [ ]:
# ============================================================
# 7. Conversion-integrity checks
# ============================================================

EXPECTED_CONTENT_MARKERS = [
    "Revenue",
    "Operating income",
    "Net income",
    "Basic earnings per share",
    "Diluted earnings per share",
    "2024",
    "2023",
    "(in € millions)"
]

source_text = page.get_text("text")

content_marker_results = {
    marker: marker.casefold() in source_text.casefold()
    for marker in EXPECTED_CONTENT_MARKERS
}

integrity_issues = []

for row, item in zip(converted_rows, row_blocks):
    lines = nonempty_lines(item["text"])

    expected_triplet = [
        row["Line Item"],
        row["2024"],
        row["2023"]
    ]

    if expected_triplet != lines:
        integrity_issues.append({
            "issue": "row_reconstruction_mismatch",
            "block_index": item["block_index"],
            "expected_lines": lines,
            "converted_values": expected_triplet
        })

duplicate_labels = len({
    row["Line Item"]
    for row in converted_rows
}) != len(converted_rows)

CONVERSION_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_sha256": SOURCE_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "source_page_count": PAGE_COUNT,
    "machine_readable_text_detected": PDF_MACHINE_READABLE,
    "conversion_method": CONVERSION_METHOD,
    "ocr_applied": False,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "converted_record_count": len(converted_rows),
    "record_count_matches_stage1_scope":
        len(converted_rows) == EXPECTED_RECORD_COUNT,
    "all_expected_content_markers_present":
        all(content_marker_results.values()),
    "expected_content_markers":
        content_marker_results,
    "duplicate_line_item_labels_detected": duplicate_labels,
    "row_reconstruction_issue_count": len(integrity_issues),
    "integrity_issues": integrity_issues,
    "manual_correction_applied": False,
    "semantic_normalisation_applied": False,
    "label_standardisation_applied": False,
    "unit_standardisation_applied": False,
    "numeric_conversion_applied": False,
    "value_modification_applied": False,
    "conversion_integrity_passed": (
        len(converted_rows) == EXPECTED_RECORD_COUNT
        and all(content_marker_results.values())
        and len(integrity_issues) == 0
    )
}

print(json.dumps(CONVERSION_INTEGRITY, indent=2, ensure_ascii=False))

if not CONVERSION_INTEGRITY["conversion_integrity_passed"]:
    raise ValueError(
        "D2 Branch B conversion failed integrity checks."
    )


In [ ]:
# ============================================================
# 8. Build the Branch B Markdown representation
# ============================================================

def escape_markdown_cell(value):
    return (
        str(value)
        .replace("\\", "\\\\")
        .replace("|", "\\|")
        .replace("\n", "<br>")
    )

header_lines = nonempty_lines(header_block["text"])
table_unit = header_lines[0]

footnote_text = " ".join(
    nonempty_lines(footnote_block["text"])
)

markdown_rows = [
    "| Line Item | 2024 | 2023 |",
    "| --- | ---: | ---: |"
]

for row in converted_rows:
    markdown_rows.append(
        "| "
        + " | ".join([
            escape_markdown_cell(row["Line Item"]),
            escape_markdown_cell(row["2024"]),
            escape_markdown_cell(row["2023"])
        ])
        + " |"
    )

STRUCTURAL_MARKDOWN = (
    "# Consolidated financial statements\n\n"
    "## Consolidated income statement\n\n"
    f"{table_unit}\n\n"
    + "\n".join(markdown_rows)
    + "\n\n"
    + footnote_text
    + "\n"
)

print(STRUCTURAL_MARKDOWN)


In [ ]:
# ============================================================
# 9. Preserve the representation and conversion evidence
# ============================================================

REPRESENTATION_PATH = (
    OUTPUT_DIR / "D2_branch_B_structural_markdown.md"
)

CONVERSION_INTEGRITY_PATH = (
    OUTPUT_DIR / "D2_branch_B_conversion_integrity.json"
)

BLOCK_AUDIT_PATH = (
    OUTPUT_DIR / "D2_branch_B_source_block_audit.json"
)

REPRESENTATION_PATH.write_text(
    STRUCTURAL_MARKDOWN,
    encoding="utf-8"
)

with open(
    CONVERSION_INTEGRITY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        CONVERSION_INTEGRITY,
        f,
        indent=2,
        ensure_ascii=False
    )

with open(
    BLOCK_AUDIT_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        block_audit,
        f,
        indent=2,
        ensure_ascii=False
    )

REPRESENTATION_SHA256 = calculate_sha256(
    REPRESENTATION_PATH
)

print("Representation:", REPRESENTATION_PATH)
print("Representation SHA-256:", REPRESENTATION_SHA256)


In [ ]:
# ============================================================
# 10. Preserve Branch B representation metadata
# ============================================================

REPRESENTATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "representation_type":
        "Structurally reconstructed Markdown table",
    "source_file": SOURCE_FILE.name,
    "source_format": SOURCE_FILE.suffix.lower(),
    "source_sha256": SOURCE_SHA256,
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "branch_name":
        BRANCH_NAME,
    "conversion_method":
        CONVERSION_METHOD,
    "llm_input_representation":
        "Structural Markdown",
    "structural_conversion_applied": True,
    "normalisation_applied": False,
    "ocr_applied": False,
    "manual_reconstruction_applied": False,
    "structural_operations": [
        "Recover machine-readable PDF text blocks",
        "Preserve physical top-to-bottom block order",
        "Identify the table header block",
        "Recover each line-item block as one row",
        "Expose the 2024/2023 column relationship in Markdown",
        "Preserve table-level unit text and footnote text"
    ],
    "operations_explicitly_not_applied": [
        "OCR",
        "Line-item rewriting",
        "Unit harmonisation",
        "Parentheses-to-negative numeric conversion during preprocessing",
        "Semantic inference",
        "Value repair"
    ],
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"]
}

REPRESENTATION_METADATA_PATH = (
    OUTPUT_DIR / "D2_branch_B_representation.json"
)

with open(
    REPRESENTATION_METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        REPRESENTATION_METADATA,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(REPRESENTATION_METADATA, indent=2, ensure_ascii=False))


## Frozen extraction controls

The extraction task and schema below are held constant with Branch A. Only representation-dependent wording changes from **original PDF** to **structurally converted Markdown**.

The Stage 1 reference values are not provided to the model.

In [ ]:
# ============================================================
# 11. Define the fixed extraction schema
# ============================================================

EXTRACTION_SCHEMA = {
    "document_id": DOCUMENT_ID,
    "record_level":
        "consolidated_income_statement_line_item",
    "fields": {
        "Line Item": {
            "type": ["string", "null"],
            "description":
                "Exact visible income-statement row label"
        },
        "Unit": {
            "type": ["string", "null"],
            "allowed_values": ALLOWED_UNITS
        },
        "Value 2024": {
            "type": ["number", "null"],
            "description":
                "Reported numerical value for 2024"
        },
        "Value 2023": {
            "type": ["number", "null"],
            "description":
                "Reported numerical value for 2023"
        }
    },
    "expected_output_structure": {
        "document_id": DOCUMENT_ID,
        "branch": BRANCH,
        "records": [
            {
                "Line Item": None,
                "Unit": None,
                "Value 2024": None,
                "Value 2023": None
            }
        ]
    }
}

print(json.dumps(EXTRACTION_SCHEMA, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 12. Operationalise the fixed Stage 1 extraction task
# ============================================================

EXTRACTION_TASK = """
You are an information extraction assistant.

Extract every line-item observation from the consolidated income
statement contained in the attached structurally converted Markdown
document.

Return one record for every visible income-statement line item.

For each record, extract:

- Line Item
- Unit
- Value 2024
- Value 2023

Extraction rules:

- Treat the attached structurally converted Markdown document as the
  only source of information.
- Extract only information explicitly supported by the document.
- Preserve each line-item label exactly as represented in the source
  representation, including footnote markers and unit text contained
  in the label.
- Preserve the association between each line item and its corresponding
  2024 and 2023 values.
- Use "EUR millions" for values governed by the table-level unit
  "(in € millions)".
- Use "EUR" for the two earnings-per-share observations.
- Convert financial values shown in parentheses into negative numerical
  values.
- Return Value 2024 and Value 2023 as numerical values.
- Do not calculate, infer, reconstruct, aggregate, correct or invent
  any value.
- Use null only when a requested value is not available.
- Do not include the table title, year headers, unit header or footnote
  explanation as separate records.
- Verify that every visible income-statement line item has been processed.
- Return only valid JSON.
- Do not include explanations before or after the JSON.
- Keep the exact field names defined in the schema.
""".strip()

print(EXTRACTION_TASK)


In [ ]:
# ============================================================
# 13. Construct and preserve the Branch B prompt
# ============================================================

EXPECTED_OUTPUT_STRUCTURE = (
    EXTRACTION_SCHEMA["expected_output_structure"]
)

FULL_PROMPT = f"""
{EXTRACTION_TASK}

Expected JSON schema:
{json.dumps(
    EXPECTED_OUTPUT_STRUCTURE,
    indent=2,
    ensure_ascii=False
)}

The structurally converted Markdown document is attached as the
extraction source.

Return only the JSON object.
""".strip()

PROMPT_PATH = OUTPUT_DIR / "D2_branch_B_prompt.txt"

PROMPT_PATH.write_text(
    FULL_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = calculate_sha256(PROMPT_PATH)

print(FULL_PROMPT)
print("Prompt SHA-256:", PROMPT_SHA256)


In [ ]:
# ============================================================
# 14. Create Branch B experiment metadata
# ============================================================

EXPERIMENT_METADATA = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_FILE.name,
    "source_format": "PDF",
    "source_sha256": SOURCE_SHA256,
    "source_page_count": PAGE_COUNT,
    "source_machine_readable": PDF_MACHINE_READABLE,
    "llm_input_representation":
        "Structural Markdown",
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "conversion_method":
        CONVERSION_METHOD,
    "conversion_integrity_file":
        CONVERSION_INTEGRITY_PATH.name,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "direct_document_ingestion": False,
    "structural_conversion_applied": True,
    "ocr_applied": False,
    "manual_correction_applied": False,
    "normalisation_applied": False,
    "semantic_harmonisation_applied": False,
    "label_standardisation_applied": False,
    "unit_standardisation_applied": False,
    "numeric_conversion_applied": False,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "expected_fields": EXPECTED_FIELDS,
    "allowed_units": ALLOWED_UNITS,
    "prompt_file": PROMPT_PATH.name,
    "prompt_sha256": PROMPT_SHA256,
    "content_validation_performed": False,
    "expected_output_format": "JSON",
    "execution_environment":
        "Independent ChatGPT conversation",
    "notes": (
        "Only the representation pathway changes relative to Branch A. "
        "The machine-readable PDF is structurally reconstructed as a "
        "Markdown table from source text blocks. No OCR, semantic "
        "normalisation, label harmonisation, unit harmonisation or "
        "numerical modification is applied. No Stage 1 reference values "
        "are supplied to the model. Content-level validation is performed "
        "separately in Validation B — D2."
    )
}

METADATA_PATH = (
    OUTPUT_DIR / "D2_branch_B_experiment_metadata.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        EXPERIMENT_METADATA,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(EXPERIMENT_METADATA, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 15. Download files required for independent LLM execution
# ============================================================

for path in [
    REPRESENTATION_PATH,
    PROMPT_PATH,
    METADATA_PATH,
    REPRESENTATION_METADATA_PATH,
    CONVERSION_INTEGRITY_PATH
]:
    files.download(path)

print(
    "\nIndependent execution instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D2_branch_B_structural_markdown.md.\n"
    "3. Submit the exact contents of D2_branch_B_prompt.txt once.\n"
    "4. Do not upload the original PDF or Stage 1 reference values.\n"
    "5. Do not manually correct, regenerate or repair the response.\n"
    "6. Save the complete response exactly as returned."
)


In [ ]:
# ============================================================
# 16. Upload the complete raw Branch B model response
# ============================================================

uploaded_output = files.upload()

if len(uploaded_output) != 1:
    raise ValueError(
        "Upload exactly one file containing the complete "
        "D2 Branch B model response."
    )

UPLOADED_RAW_OUTPUT = Path(next(iter(uploaded_output)))

print("Uploaded raw response:", UPLOADED_RAW_OUTPUT)


In [ ]:
# ============================================================
# 17. Preserve the raw model response unchanged
# ============================================================

RAW_RESPONSE_PATH = (
    OUTPUT_DIR / "D2_branch_B_raw_response.txt"
)

raw_response_text = UPLOADED_RAW_OUTPUT.read_text(
    encoding="utf-8"
)

RAW_RESPONSE_PATH.write_text(
    raw_response_text,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = calculate_sha256(
    RAW_RESPONSE_PATH
)

print("Raw response preserved.")
print("Raw response SHA-256:", RAW_RESPONSE_SHA256)


In [ ]:
# ============================================================
# 18. Parse raw response without modifying it
# ============================================================

valid_json = False
json_parsing_error = None
raw_extraction = None

try:
    raw_extraction = json.loads(raw_response_text)
    valid_json = True
except json.JSONDecodeError as error:
    json_parsing_error = str(error)

print("Valid JSON:", valid_json)

if json_parsing_error:
    print("Parsing error:", json_parsing_error)


In [ ]:
# ============================================================
# 19. Validate top-level output structure
# ============================================================

top_level_object_valid = (
    valid_json
    and isinstance(raw_extraction, dict)
)

document_id_present = (
    top_level_object_valid
    and "document_id" in raw_extraction
)

document_id_correct = (
    document_id_present
    and raw_extraction.get("document_id") == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch" in raw_extraction
)

branch_correct = (
    branch_present
    and raw_extraction.get("branch") == BRANCH
)

records_present = (
    top_level_object_valid
    and "records" in raw_extraction
)

records_is_list = (
    records_present
    and isinstance(raw_extraction.get("records"), list)
)

records = (
    raw_extraction["records"]
    if records_is_list
    else []
)

print({
    "top_level_object_valid": top_level_object_valid,
    "document_id_present": document_id_present,
    "document_id_correct": document_id_correct,
    "branch_present": branch_present,
    "branch_correct": branch_correct,
    "records_present": records_present,
    "records_is_list": records_is_list
})


In [ ]:
# ============================================================
# 20. Validate individual record schemas
# ============================================================

record_structure_issues = []

for record_index, record in enumerate(records):
    issues = []

    if not isinstance(record, dict):
        issues.append("Record is not a JSON object.")
    else:
        actual_fields = set(record.keys())
        expected_fields = set(EXPECTED_FIELDS)

        missing_fields = sorted(
            expected_fields - actual_fields
        )
        additional_fields = sorted(
            actual_fields - expected_fields
        )

        if missing_fields:
            issues.append({
                "missing_fields": missing_fields
            })

        if additional_fields:
            issues.append({
                "additional_fields": additional_fields
            })

    if issues:
        record_structure_issues.append({
            "record_index": record_index,
            "issues": issues
        })

records_with_structure_issues = len(
    record_structure_issues
)

print(
    "Records with structure issues:",
    records_with_structure_issues
)


In [ ]:
# ============================================================
# 21. Validate extracted field types and allowed units
# ============================================================

field_type_issues = []
unit_issues = []

for record_index, record in enumerate(records):
    if not isinstance(record, dict):
        continue

    line_item = record.get("Line Item")
    unit = record.get("Unit")

    if line_item is not None and not isinstance(line_item, str):
        field_type_issues.append({
            "record_index": record_index,
            "field": "Line Item",
            "observed_type": type(line_item).__name__
        })

    if unit is not None and not isinstance(unit, str):
        field_type_issues.append({
            "record_index": record_index,
            "field": "Unit",
            "observed_type": type(unit).__name__
        })

    if unit is not None and unit not in ALLOWED_UNITS:
        unit_issues.append({
            "record_index": record_index,
            "observed_unit": unit
        })

    for field in NUMERIC_FIELDS:
        value = record.get(field)

        if value is None:
            continue

        if isinstance(value, bool) or not isinstance(value, (int, float)):
            field_type_issues.append({
                "record_index": record_index,
                "field": field,
                "observed_type": type(value).__name__,
                "observed_value": value
            })

records_with_type_issues = len({
    issue["record_index"]
    for issue in field_type_issues
})

print("Records with type issues:", records_with_type_issues)
print("Unexpected units:", len(unit_issues))


In [ ]:
# ============================================================
# 22. Check record count, duplicate identities and missing values
# ============================================================

record_count = len(records)

record_count_valid = (
    record_count == EXPECTED_RECORD_COUNT
)

line_items = [
    record.get("Line Item")
    for record in records
    if isinstance(record, dict)
]

non_null_line_items = [
    item
    for item in line_items
    if item is not None
]

duplicate_line_items = sorted({
    item
    for item in non_null_line_items
    if non_null_line_items.count(item) > 1
})

missing_values_by_field = {
    field: 0
    for field in EXPECTED_FIELDS
}

for record in records:
    if not isinstance(record, dict):
        continue

    for field in EXPECTED_FIELDS:
        if (
            field not in record
            or record.get(field) is None
        ):
            missing_values_by_field[field] += 1

print("Expected records:", EXPECTED_RECORD_COUNT)
print("Observed records:", record_count)
print("Record count valid:", record_count_valid)
print("Duplicate line items:", duplicate_line_items)
print("Missing values:", missing_values_by_field)


In [ ]:
# ============================================================
# 23. Create Branch B structural/schema diagnostics
# ============================================================

record_schema_valid = (
    records_with_structure_issues == 0
)

field_types_valid = (
    records_with_type_issues == 0
)

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid,
    field_types_valid
])

TECHNICAL_DIAGNOSTICS = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "valid_json": bool(valid_json),
    "json_parsing_error": json_parsing_error,
    "top_level_object_valid": bool(top_level_object_valid),
    "document_id_present": bool(document_id_present),
    "document_id_correct": bool(document_id_correct),
    "branch_present": bool(branch_present),
    "branch_correct": bool(branch_correct),
    "top_level_records_present": bool(records_present),
    "records_is_list": bool(records_is_list),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,
    "number_of_records":
        int(record_count),
    "record_count_valid":
        bool(record_count_valid),

    "records_with_structure_issues":
        int(records_with_structure_issues),
    "record_structure_issues":
        record_structure_issues,

    "records_with_type_issues":
        int(records_with_type_issues),
    "field_type_issues":
        field_type_issues,

    "records_with_unexpected_units":
        int(len(unit_issues)),
    "unit_issues":
        unit_issues,

    "duplicate_line_item_count":
        int(len(duplicate_line_items)),
    "duplicate_line_items":
        duplicate_line_items,

    "missing_values_by_field":
        missing_values_by_field,

    "record_schema_valid":
        bool(record_schema_valid),
    "field_types_valid":
        bool(field_types_valid),

    "structurally_evaluable":
        bool(structurally_evaluable)
}

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR / "D2_branch_B_technical_diagnostics.json"
)

with open(
    TECHNICAL_DIAGNOSTICS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        TECHNICAL_DIAGNOSTICS,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(TECHNICAL_DIAGNOSTICS, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 24. Preserve parsed extraction only when JSON is valid
# ============================================================

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR / "D2_branch_B_parsed_extraction.json"
)

if valid_json:
    with open(
        PARSED_EXTRACTION_PATH,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            raw_extraction,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("Parsed extraction saved:", PARSED_EXTRACTION_PATH)
else:
    print(
        "Parsed extraction was not created because "
        "the preserved response is invalid JSON."
    )


In [ ]:
# ============================================================
# 25. Create Branch B experiment summary
# ============================================================

EXPERIMENT_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_FILE.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified":
        SOURCE_HASH_MATCH and PAGE_COUNT == EXPECTED_PAGE_COUNT,
    "llm_input_representation":
        "Structural Markdown",
    "representation_file":
        REPRESENTATION_PATH.name,
    "representation_sha256":
        REPRESENTATION_SHA256,
    "conversion_method":
        CONVERSION_METHOD,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,
    "content_validation_performed":
        False,
    "structural_conversion_applied": True,
    "semantic_normalisation_applied": False,
    "manual_correction_applied": False,
    "ocr_applied": False,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": int(record_count),
    "valid_json": bool(valid_json),
    "structurally_evaluable": bool(structurally_evaluable),
    "record_count_valid": bool(record_count_valid),
    "records_with_structure_issues":
        int(records_with_structure_issues),
    "records_with_type_issues":
        int(records_with_type_issues),
    "records_with_unexpected_units":
        int(len(unit_issues)),
    "duplicate_line_item_count":
        int(len(duplicate_line_items)),
    "raw_response_preserved": True,
    "raw_response_sha256": RAW_RESPONSE_SHA256,
    "parsed_extraction_created": bool(valid_json),
    "notes":
        "Content-level validation is performed separately in Validation B — D2."
}

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR / "D2_branch_B_experiment_summary.json"
)

with open(
    EXPERIMENT_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        EXPERIMENT_SUMMARY,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(EXPERIMENT_SUMMARY, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 26. List and download generated outputs
# ============================================================

generated_outputs = [
    REPRESENTATION_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_METADATA_PATH,
    BLOCK_AUDIT_PATH,
    PROMPT_PATH,
    METADATA_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if valid_json:
    generated_outputs.append(
        PARSED_EXTRACTION_PATH
    )

print("Generated outputs:")
for path in generated_outputs:
    print("-", path.name)

for path in generated_outputs:
    files.download(path)
